# HeartTwin ARGO — resumable recovery

**Use a CPU runtime and Run all.** No GPU, token, prior ZIP upload or manual package setup is needed. Allow Drive access. The notebook downloads immutable repaired source and installs an isolated environment. It requires the standard Colab pip (22.3 or newer) and changes no global packages.

The original run paused after 9,312 of 11,821 selected files, before record validation. Its parser would also reject valid non-AVP annotations. This recovery reuses and extends the verified raw cache at `HeartTwinValidation/09_HeartTwin_ARGO_v1/data` if present. It keeps reports and checkpoints in a separate v3 folder. The v2 run failed during `ensurepip`, before validation. This version creates the environment without `ensurepip` and uses Colab’s existing pip to install into that isolated environment; it checks dependency consistency and imports before starting the audit. With no v1 cache it downloads all required files from scratch. **Do not delete your v1 folder.**

Each record is checked and saved immediately. The 110-minute session budget permits safe continuation on another runtime: rerun this same notebook with the same run label and Drive account. Completed results and raw files are hash-checked before reuse. Small checkpoint pages avoid growing global rescans. Source, protocol and numerical-environment changes are rejected instead of silently mixing results. Network speed may dominate; T4 offers no benefit for this audit.

All 1,962 records / nine patients are required for `READINESS_ONLY`. `PAUSED` is incomplete, even if the cell exits without an exception. The ZIP downloads on completion, pause or caught failure; upload it here. Hard runtime termination cannot trigger a download, but Drive retains the saved data and record checkpoints for the next run. Raw ECG/EGM files stay in the data cache and are excluded from the ZIP.

Class A has AVP boundaries; P/U use one out-of-range note. Unknown remains unknown; the sentinel is not converted to a real time. Expert AVP bounds are **not LAT truth**. Only the final beat is documented reliable. Coordinate units remain unconfirmed, and paired volumetric scar/fibers, mechanics/hemodynamics and longitudinal outcome qualification are missing. This performs ingestion readiness, not model accuracy or clinical validation.

Data: [ARGO 1.0.0](https://physionet.org/content/argo/1.0.0/), DOI [10.13026/8gh2-e660](https://doi.org/10.13026/8gh2-e660), CC BY-NC-SA 4.0. All required inputs are verified against its pinned publisher checksum manifest.


In [ ]:
# CPU is sufficient. Leave SMOKE_RECORD_LIMIT=None for the full audit.
USE_DRIVE = True
RUN_LABEL = "09_HeartTwin_ARGO_v3_recovery"
BUDGET_SECONDS = 6600
SMOKE_RECORD_LIMIT = None
SOURCE_COMMIT = "1634ef7eb7f4e632cc11dbcd4066df23bfe8066e"
SOURCE_SHA256 = "d7354308f45b097f393ab48a1cfb361f9609a39da608046d9c073d28f529e637"

import hashlib, json, os, re, shutil, subprocess, sys, time, traceback, urllib.request, zipfile
from pathlib import Path
from google.colab import drive, files

CONTENT = Path("/content")
ROOT = CONTENT / RUN_LABEL
REQUIREMENTS = ["numpy==2.2.6", "scipy==1.15.3", "pandas==2.2.3", "wfdb==4.2.0",
                "matplotlib==3.10.1", "pydantic==2.11.7", "PyYAML==6.0.2", "requests==2.32.4"]
ENV = dict(os.environ, MPLBACKEND="Agg", PYTHONNOUSERSITE="1", OPENBLAS_NUM_THREADS="1",
           OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
started = time.monotonic()
error = None
can_write = True

def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""): h.update(block)
    return h.hexdigest()

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(exist_ok=True, parents=True)
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n")
    os.replace(temp, path)

def command(args, log, cwd=None, timeout=1800):
    log.parent.mkdir(exist_ok=True, parents=True)
    with log.open("w") as f:
        f.write("COMMAND " + json.dumps(list(map(str,args))) + "\n"); f.flush()
        result = subprocess.run(list(map(str,args)), cwd=cwd, env=ENV, stdout=f,
                                stderr=subprocess.STDOUT, timeout=timeout)
    if result.returncode:
        print(log.read_text()[-10000:])
        raise RuntimeError("Command failed; see " + log.name + " in the exported ZIP")

def source_download(path):
    if path.exists():
        if sha(path) != SOURCE_SHA256: raise ValueError("Cached source checksum mismatch")
        return
    path.parent.mkdir(exist_ok=True, parents=True)
    temp = path.with_suffix(".part")
    for attempt in range(4):
        try:
            with urllib.request.urlopen("https://codeload.github.com/Virelion-Biotech/Virelion-HeartTwin/zip/" + SOURCE_COMMIT, timeout=60) as r, temp.open("wb") as f:
                shutil.copyfileobj(r,f)
            if sha(temp) != SOURCE_SHA256: raise ValueError("Downloaded source checksum mismatch")
            os.replace(temp,path); return
        except Exception:
            if attempt == 3: raise
            time.sleep(2 ** attempt)

def export_zip():
    paths = [p for p in ROOT.rglob("*") if p.is_file() and p.relative_to(ROOT).parts[0] not in {"cache","data"}
             and p.name != "ARTIFACT_HASHES.json" and not p.name.endswith((".tmp",".part"))]
    atomic_json(ROOT / "ARTIFACT_HASHES.json", {str(p.relative_to(ROOT)): sha(p) for p in paths})
    paths.append(ROOT / "ARTIFACT_HASHES.json")
    output = CONTENT / (RUN_LABEL + "_results.zip"); temp = output.with_suffix(".tmp")
    with zipfile.ZipFile(temp,"w",zipfile.ZIP_DEFLATED) as z:
        for p in sorted(paths): z.write(p,str(p.relative_to(ROOT)))
    os.replace(temp,output)
    with zipfile.ZipFile(output) as z:
        hashes = json.loads(z.read("ARTIFACT_HASHES.json")); names=z.namelist()
        assert len(names)==len(set(names)) and set(names)==set(hashes)|{"ARTIFACT_HASHES.json"}
        for name,h in hashes.items(): assert hashlib.sha256(z.read(name)).hexdigest()==h
    print("Results ZIP:",output); files.download(str(output))

try:
    if not re.fullmatch(r"[A-Za-z0-9_-]+",RUN_LABEL): raise ValueError("Use a simple RUN_LABEL")
    if USE_DRIVE:
        drive.mount(str(CONTENT / "drive"))
        ROOT = CONTENT / "drive/MyDrive/HeartTwinValidation" / RUN_LABEL
    ROOT.mkdir(exist_ok=True,parents=True)
    protocol = {"run_label":RUN_LABEL,"source_commit":SOURCE_COMMIT,"source_sha256":SOURCE_SHA256,
                "requirements":REQUIREMENTS,"scope":"Resumable ARGO ingestion readiness only",
                "smoke_record_limit":SMOKE_RECORD_LIMIT,"bootstrap":"host-pip-target-venv-v1"}
    if (ROOT / "protocol.json").exists() and json.loads((ROOT / "protocol.json").read_text()) != protocol:
        can_write=False; raise ValueError("Different protocol in this folder; choose a new RUN_LABEL")
    atomic_json(ROOT / "protocol.json",protocol)
    atomic_json(ROOT / "RUN_STATUS.json",{"status":"RUNNING"})
    if sys.version_info[:2] not in {(3,11),(3,12),(3,13)}:
        raise RuntimeError("Choose a Python 3.11–3.13 Colab runtime for these dependency pins")
    legacy = CONTENT / "drive/MyDrive/HeartTwinValidation/09_HeartTwin_ARGO_v1/data"
    data = legacy if USE_DRIVE and legacy.is_dir() else ROOT / "data"
    print("Data cache:",data)
    print("Each required cached file is rehashed; old receipts alone are never trusted.")
    atomic_json(ROOT / "CACHE_LOCATION.json",{"data_directory":str(data),"reuse_v1":data==legacy})
    archive = ROOT / "cache/source.zip"; source_download(archive)
    runtime = CONTENT / (RUN_LABEL + "_runtime"); runtime.mkdir(exist_ok=True,parents=True)
    with zipfile.ZipFile(archive) as z:
        for info in z.infolist():
            if not (runtime / info.filename).resolve().is_relative_to(runtime.resolve()):
                raise ValueError("Unsafe source archive path")
        z.extractall(runtime)
    repo = runtime / ("Virelion-HeartTwin-" + SOURCE_COMMIT)
    atomic_json(ROOT / "SOURCE_SNAPSHOT.json",{"commit":SOURCE_COMMIT,"archive_sha256":sha(archive),
                "files":{str(p.relative_to(repo)):sha(p) for p in sorted((repo / "hearttwin").glob("*.py"))}})
    venv = runtime / "venv"
    # Colab may not provide a working ensurepip. Use host pip's documented
    # --python support to seed an isolated environment without invoking ensurepip.
    command([sys.executable,"-m","venv","--without-pip",venv], ROOT / "logs/venv.log")
    python = venv / "bin/python"
    command([sys.executable,"-m","pip","--python",python,"install",
             "pip==25.1.1",*REQUIREMENTS], ROOT / "logs/install.log")
    command([python,"-m","pip","check"], ROOT / "logs/pip-check.log")
    command([python,"-c","import sys; assert sys.prefix != sys.base_prefix; "
             "import numpy, scipy, pandas, wfdb, matplotlib, pydantic, yaml, requests; "
             "print(sys.version); print(sys.executable)"], ROOT / "logs/imports.log")
    ENV["PYTHONPATH"] = str(repo)
    with (ROOT / "environment.txt").open("w") as f:
        subprocess.run([str(python),"-m","pip","freeze"],env=ENV,stdout=f,check=True)
    remaining = max(0, BUDGET_SECONDS - (time.monotonic()-started))
    args=[python,"-m","hearttwin.argo_readiness","--data",data,"--output",ROOT / "validation",
          "--budget-seconds",str(remaining)]
    if SMOKE_RECORD_LIMIT is not None: args += ["--max-records",str(SMOKE_RECORD_LIMIT)]
    command(args, ROOT / "logs/validation.log",cwd=repo,timeout=remaining+300)
    result=json.loads((ROOT / "validation/RUN_STATUS.json").read_text())
    if result["status"] not in {"READINESS_ONLY","PAUSED"}: raise RuntimeError("Unexpected worker status")
    atomic_json(ROOT / "RUN_STATUS.json",{**result,"seconds_this_session":time.monotonic()-started,
                "source_commit":SOURCE_COMMIT,"data_directory":str(data)})
    if (data / "SHA256SUMS.txt").exists():
        if sha(data / "SHA256SUMS.txt") != "0bc4fd8d597951b992249a1be080365ecefb41582f87108dc933d3bade808141":
            raise ValueError("Publisher manifest mismatch")
        shutil.copyfile(data / "SHA256SUMS.txt", ROOT / "PUBLISHER_SHA256SUMS.txt")
    print("Status:",result["status"],"Records verified:",result["records_verified"],"/1962")
    if result["status"]=="PAUSED": print("Progress saved. Run this same notebook again on a new CPU runtime to continue.")
    else: print("All ingestion checks completed. LAT accuracy, paired patient-twin and clinical qualification remain unestablished.")
except BaseException:
    error=traceback.format_exc();print(error)
    if can_write:
        ROOT.mkdir(exist_ok=True,parents=True)
        atomic_json(ROOT / "RUN_STATUS.json",{"status":"FAILED","error":error,"source_commit":SOURCE_COMMIT})
finally:
    if ROOT.exists() and can_write: export_zip()
if error: raise RuntimeError("Run failed; upload the exported ZIP for diagnosis")
